In [1]:
import os
import sys

os.chdir("..")
sys.path.append(os.getcwd())
import json
import torch
import numpy as np
import pandas as pd

from models import get_model
from datasets import get_dataloaders
from ensemble import evaluate_ensemble, evaluate_stacking, save_report
from utils import set_seed

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RESULTS_DIR = "./results/exp1_hparams"
TOP_K = 5

In [2]:
def load_results(results_dir):
    rows = []

    for file in os.listdir(results_dir):
        if not file.endswith(".json"):
            continue

        with open(os.path.join(results_dir, file)) as f:
            data = json.load(f)

        config = data["config"]
        exp_id = int(file.split("_")[1])

        for seed_idx, run in enumerate(data["runs"]):
            rows.append({
                "exp_id": exp_id,
                "seed": seed_idx,
                "model": config["model"],
                "dropout": config.get("dropout", 0),

                "model_name": f"model_{exp_id}_{config['model']}_{seed_idx}.pth",

                "val_acc": run["val"]["accuracy"],
                "test_acc": run["test"]["accuracy"],
                "test_macro_f1": run["test"]["macro_f1"],
            })

    return pd.DataFrame(rows)


df = load_results(RESULTS_DIR)
df.head()

,exp_id,seed,model,dropout,model_name,val_acc,test_acc,test_macro_f1
0,20,0,resnet18,0.3,model_20_resnet18_0.pth,0.898422,0.897044,0.896930
1,20,1,resnet18,0.3,model_20_resnet18_1.pth,0.898122,0.896533,0.896497
2,20,2,resnet18,0.3,model_20_resnet18_2.pth,0.898322,0.895700,0.895654
3,13,0,cnn,0.5,model_13_cnn_0.pth,0.821267,0.818667,0.818444
4,13,1,cnn,0.5,model_13_cnn_1.pth,0.822344,0.819067,0.818792


In [3]:
top_df = df.sort_values("test_macro_f1", ascending=False).head(TOP_K)

top_df[["model_name", "test_macro_f1"]]

,model_name,test_macro_f1
109,model_33_resnet18_1.pth,0.900002
33,model_22_resnet18_0.pth,0.899972
97,model_24_resnet18_1.pth,0.899409
106,model_32_resnet18_1.pth,0.899282
26,model_25_resnet18_2.pth,0.899214


In [4]:
def load_models(df):
    models = []
    weights = []

    for _, row in df.iterrows():
        path = os.path.join(RESULTS_DIR, row["model_name"])

        model = get_model(row["model"], row["dropout"])
        model.load_state_dict(torch.load(path, map_location=device))
        model.to(device)
        model.eval()

        models.append(model)
        weights.append(row["val_acc"])

    return models, weights


models, weights = load_models(top_df)

print("Loaded models:", len(models))

Loaded models: 5


In [5]:
_, val_loader, test_loader = get_dataloaders(
    batch_size=128,
    use_augmentation=False,
    model_name="resnet18" 
)

In [11]:
hard_stats = evaluate_ensemble(
    models,
    test_loader,
    device,
    method="hard"
)

print("Hard voting:", hard_stats["macro_f1"])

TypeError: evaluate_ensemble() got an unexpected keyword argument 'method'

In [ ]:
soft_stats = evaluate_ensemble(models, test_loader, device, method="soft")
print("Soft voting:", soft_stats["macro_f1"])

Soft: 0.9105678424460664


In [ ]:
weighted_stats = evaluate_ensemble(models, test_loader, device, weights=weights, method="weighted")
print("Weighted voting:", weighted_stats["macro_f1"])

Weighted: 0.9105678424460664


In [ ]:
stacking_results = {}
    
for model_type in ["logreg", "ridge", "rf", "gb", "svm","knn"]:
    stats = evaluate_stacking(models, val_loader, test_loader, device, model_type)
    stacking_results[model_type] = stats
    print("Stacking:", model_type, stats["macro_f1"])

logreg 0.9104815838174453
ridge 0.9107153162064936
rf 0.9107202387699911
gb 0.9070767781502307
svm 0.9109032389213934


In [ ]:
stats = evaluate_stacking(models, val_loader, test_loader, device, "knn" )
stacking_results["knn"] = stats
print("knn", stats["macro_f1"])

knn 0.901152762186836


In [10]:
comparison = pd.DataFrame([
    {"method": "single", "macro_f1": evaluate_ensemble([models[0]], test_loader, device)["accuracy"]},
    {"method": "soft", "macro_f1": soft_stats["macro_f1"]},
    {"method": "weighted", "macro_f1": weighted_stats["macro_f1"]},
    {"method": "hard", "macro_f1": hard_stats["macro_f1"]},
    *[{"method": f"stacking_{k}", "macro_f1": v} for k, v in stacking_results.items()]
])

comparison

NameError: name 'hard_stats' is not defined

In [ ]:
save_report("./results/final_ensemble.json", {
    "soft": soft_stats,
    "weighted": weighted_stats,
    "stacking": stacking_results,
    "models": top_df["model_name"].tolist()
})